# 04 — MCP + local RAG + live web

**Learning goal:** extend the lazy MCP-RAG server with current-date grounding, live search, safe page fetching, and an agent policy that separates local knowledge from current web information.

**Prerequisites:** Python 3 and this notebook inside `demos/`. Run cells top to bottom and choose `PROFILE="onia"` or `"devtalks"`. Installation, configuration, tool registration, MCP discovery, `current_date`, and `list_sources` are offline. `search_knowledge` requires LM Studio embeddings; the agent requires LM Studio chat and may also require embeddings/internet according to its choices. `search_web` and `fetch_page` require internet. All live/model paths are opt-in and disabled by default. Each factory run creates a fresh server, so re-running cells does not duplicate tools.

In [ ]:
%pip install -q openai==2.53.0 chromadb==1.5.9 "mcp[cli]==2.0.0" ddgs==9.14.4

## 1. Configuration and profile
Edit constants or override them with same-named environment variables. Document resolution starts from `Path.cwd()` and reports how to fix a wrong launch directory.

In [ ]:
import os
from pathlib import Path

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "lm-studio")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen/qwen3.5-9b")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-qwen3-embedding-4b")
PROFILE = os.getenv("PROFILE", "devtalks").lower()
TOP_K = int(os.getenv("TOP_K", "3"))
RUN_WEB_DEMO = False       # internet only
RUN_AGENTIC_DEMO = False  # LM Studio; tools may also use internet/embeddings

if PROFILE not in {"onia", "devtalks"}:
    raise ValueError("PROFILE must be 'onia' or 'devtalks'.")
if TOP_K <= 0:
    raise ValueError("TOP_K must be a positive integer.")

def find_demo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd / "demos", *cwd.parents]:
        if (candidate / "documents" / "shared" / "mcp_plus_rag.md").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find demos/documents. Launch Jupyter from the presentations root or demos directory."
    )

DEMO_ROOT = find_demo_root()
DOCUMENTS_ROOT = DEMO_ROOT / "documents"
print(f"Profile: {PROFILE} | documents: {DOCUMENTS_ROOT}")

## 2. Profile-aware local corpus — **offline**
The local side retains the same MCP-RAG profile split; web tools supplement rather than replace these curated documents.

In [ ]:
PROFILE_DOCUMENTS = {
    "onia": [
        "shared/lm_studio.md", "shared/mcp_overview.md",
        "shared/mcp_plus_rag.md", "onia/qwen_models_mcp_rag.md",
        "onia/onia_conference.md",
    ],
    "devtalks": [
        "shared/lm_studio.md", "shared/mcp_overview.md",
        "shared/mcp_plus_rag.md", "shared/qwen_models.md",
        "devtalks/devtalks_conference.md",
    ],
}
selected_paths = [DOCUMENTS_ROOT / relative for relative in PROFILE_DOCUMENTS[PROFILE]]
missing = [path for path in selected_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing demo documents: {missing}")
print("Local sources:", [path.relative_to(DOCUMENTS_ROOT).as_posix() for path in selected_paths])

## 3. Safe URL validation for page fetching
Before any fetch, the URL must use HTTP(S), contain no credentials, and resolve only to public addresses. Loopback, private, link-local, reserved, multicast, and unspecified targets are rejected to reduce server-side request forgery risk. DNS resolution occurs only when `fetch_page` is actually called.

In [ ]:
import ipaddress
import socket
from urllib.parse import urlparse

def validate_public_http_url(url: str) -> str:
    parsed = urlparse(url.strip())
    if parsed.scheme.lower() not in {"http", "https"}:
        raise ValueError("Only http:// and https:// URLs may be fetched.")
    if not parsed.hostname:
        raise ValueError("URL must include a hostname.")
    if parsed.username is not None or parsed.password is not None:
        raise ValueError("URLs containing credentials are not allowed.")
    try:
        port = parsed.port or (443 if parsed.scheme.lower() == "https" else 80)
        addresses = {
            info[4][0] for info in socket.getaddrinfo(parsed.hostname, port, type=socket.SOCK_STREAM)
        }
    except (socket.gaierror, ValueError) as exc:
        raise ValueError(f"Could not resolve a safe destination for {url!r}.") from exc
    if not addresses:
        raise ValueError("Hostname did not resolve to an address.")
    for address in addresses:
        ip = ipaddress.ip_address(address)
        if (ip.is_private or ip.is_loopback or ip.is_link_local or ip.is_reserved
                or ip.is_multicast or ip.is_unspecified):
            raise ValueError(f"Refusing non-public destination: {address}")
    return parsed.geturl()

print("URL validator ready; no network request made.")

## 4. Create the lazy MCP-RAG + web server
The local index is built only by `search_knowledge`. `search_web` returns snippets, while `fetch_page` validates and extracts a selected page. `current_date` grounds relative dates and `list_sources` exposes the local manifest without model calls.

In [ ]:
from datetime import datetime
from typing import Any
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from ddgs import DDGS
from mcp.server import MCPServer
from openai import OpenAI

class OpenAIEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, client: OpenAI, model: str) -> None:
        self.client = client
        self.model = model

    def __call__(self, input: Documents) -> Embeddings:
        response = self.client.embeddings.create(model=self.model, input=list(input))
        return [item.embedding for item in response.data]

def make_web_rag_server() -> MCPServer:
    server = MCPServer(f"notebook-mcp-rag-web-{PROFILE}")
    state: dict[str, Any] = {"collection": None, "chroma_client": None}

    def ensure_index():
        if state["collection"] is not None:
            return state["collection"]
        llm = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
        chroma_client = chromadb.EphemeralClient()
        collection = chroma_client.get_or_create_collection(
            name=f"notebook_mcp_rag_web_{PROFILE}",
            embedding_function=OpenAIEmbeddingFunction(llm, EMBEDDING_MODEL),
        )
        sources = [path.relative_to(DOCUMENTS_ROOT).as_posix() for path in selected_paths]
        collection.add(
            ids=[f"doc-{index}" for index in range(len(selected_paths))],
            documents=[path.read_text(encoding="utf-8") for path in selected_paths],
            metadatas=[{"source": source} for source in sources],
        )
        state.update(collection=collection, chroma_client=chroma_client)
        print(f"Lazy local index built with {collection.count()} documents.")
        return collection

    @server.tool()
    def search_knowledge(query: str, k: int = TOP_K) -> str:
        """Search the selected local demo corpus; use for stable, local information."""
        if k <= 0:
            return "k must be a positive integer."
        collection = ensure_index()
        result = collection.query(query_texts=[query], n_results=min(k, collection.count()))
        return "\n\n---\n\n".join(
            f"[source: {meta['source']}] (distance={distance:.4f})\n{text.strip()}"
            for text, meta, distance in zip(
                result["documents"][0], result["metadatas"][0], result["distances"][0]
            )
        )

    @server.tool()
    def search_web(query: str, k: int = 5) -> str:
        """Search the live web for current information and return result snippets and URLs."""
        if k <= 0:
            return "k must be a positive integer."
        try:
            results = DDGS().text(query, max_results=min(k, 10))
        except Exception as exc:
            return f"Web search failed: {exc}"
        if not results:
            return "No web results found."
        return "\n\n".join(
            f"{index}. {item.get('title', '').strip()}\nURL: {item.get('href', '').strip()}\n{item.get('body', '').strip()}"
            for index, item in enumerate(results, 1)
        )

    @server.tool()
    def fetch_page(url: str, max_chars: int = 6000) -> str:
        """Validate and fetch a public HTTP(S) page when a search snippet is insufficient."""
        if not 1 <= max_chars <= 20000:
            return "max_chars must be between 1 and 20000."
        try:
            safe_url = validate_public_http_url(url)
        except ValueError as exc:
            return f"URL rejected: {exc}"
        try:
            extracted = DDGS().extract(safe_url)
        except Exception as exc:
            return f"Failed to fetch {safe_url}: {exc}"
        content = extracted.get("content", "") if isinstance(extracted, dict) else str(extracted)
        if isinstance(content, bytes):
            content = content.decode("utf-8", errors="replace")
        content = content.strip()
        if not content:
            return f"No readable content extracted from {safe_url}."
        suffix = "\n\n[… truncated …]" if len(content) > max_chars else ""
        return f"Content of {safe_url}:\n\n{content[:max_chars].rstrip()}{suffix}"

    @server.tool()
    def current_date(timezone: str = "UTC") -> str:
        """Return today's date in an IANA timezone for relative-date questions."""
        try:
            now = datetime.now(ZoneInfo(timezone))
        except ZoneInfoNotFoundError:
            return f"Unknown timezone: {timezone!r}"
        return f"{now:%Y-%m-%d (%A)}"

    @server.tool()
    def list_sources() -> list[str]:
        """List local profile sources without building the vector index."""
        return [path.relative_to(DOCUMENTS_ROOT).as_posix() for path in selected_paths]

    return server

web_rag_server = make_web_rag_server()
print("Fresh server created; no model or web calls made.")

## 5. Discover tools through MCP — **offline**
The direct protocol demonstration lists all tools and safely calls only the date and source tools. Jupyter top-level `await` keeps the client/server exchange concise.

In [ ]:
from mcp import ClientSession
from mcp.client._memory import InMemoryTransport

def tool_result_text(result: Any) -> str:
    return "\n".join(getattr(block, "text", str(block)) for block in result.content).strip()

async with InMemoryTransport(web_rag_server, raise_exceptions=True) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        print("Advertised:", [tool.name for tool in tools.tools])
        date_result = await session.call_tool("current_date", {"timezone": "Europe/Bucharest"})
        source_result = await session.call_tool("list_sources", {})
        print("Date:", tool_result_text(date_result))
        print("Sources:", tool_result_text(source_result))

## 6. Optional direct web exploration — **requires internet**
Enable `RUN_WEB_DEMO` to search. To demonstrate full-page extraction, copy a public HTTP(S) result URL into `WEB_PAGE_URL`; the fetch tool will validate it first. LM Studio is not used in this cell.

In [ ]:
WEB_QUERY = "latest official MCP specification news"
WEB_PAGE_URL = ""  # paste a public URL returned by search_web

if RUN_WEB_DEMO:
    direct_web_server = make_web_rag_server()
    async with InMemoryTransport(direct_web_server, raise_exceptions=True) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            search_result = await session.call_tool("search_web", {"query": WEB_QUERY, "k": 3})
            print(tool_result_text(search_result))
            if WEB_PAGE_URL:
                page_result = await session.call_tool("fetch_page", {"url": WEB_PAGE_URL})
                print("\nFetched page:\n", tool_result_text(page_result)[:2000])
else:
    print("Skipped live web calls. Set RUN_WEB_DEMO=True to enable them.")

## 7. Optional local/web agent — **requires LM Studio; may require internet**
The policy routes stable demo facts to local RAG and current facts to the web. It asks for the date before relative-date searches, fetches promising pages when snippets are insufficient, and requires filename or URL citations.

In [ ]:
import json

def openai_tool_schemas(mcp_tools: Any) -> list[dict]:
    return [{
        "type": "function",
        "function": {
            "name": tool.name, "description": tool.description or "",
            "parameters": tool.inputSchema or {"type": "object"},
        },
    } for tool in mcp_tools.tools]

QUESTION = (
    "What is ONIA according to the local corpus, and what is the latest news on its official site?"
    if PROFILE == "onia"
    else "What is DevTalks according to the local corpus, and what is the latest news on its official site?"
)

if RUN_AGENTIC_DEMO:
    llm = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    agent_server = make_web_rag_server()
    async with InMemoryTransport(agent_server, raise_exceptions=True) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            schemas = openai_tool_schemas(await session.list_tools())
            messages = [
                {"role": "system", "content": (
                    "Use search_knowledge for stable facts about this demo and its local profile. "
                    "Use search_web for current information; call current_date first for relative dates. "
                    "Web results are snippets, so call fetch_page on a promising URL when needed. "
                    "Answer only from retrieved material. Cite local facts by filename and web facts by URL; "
                    "if evidence is missing, say so."
                )},
                {"role": "user", "content": QUESTION},
            ]
            for step in range(8):
                response = llm.chat.completions.create(
                    model=CHAT_MODEL, messages=messages, tools=schemas,
                    tool_choice="auto", temperature=0.2,
                )
                message = response.choices[0].message
                calls = message.tool_calls or []
                if not calls:
                    print(message.content or "")
                    break
                messages.append({"role": "assistant", "content": message.content or "",
                                 "tool_calls": [call.model_dump() for call in calls]})
                for call in calls:
                    arguments = json.loads(call.function.arguments or "{}")
                    result = await session.call_tool(call.function.name, arguments)
                    text = tool_result_text(result)
                    print(f"tool: {call.function.name}({arguments}) -> {text[:180]}…")
                    messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
            else:
                print("Stopped after eight tool-selection steps.")
else:
    print("Skipped agent. Set RUN_AGENTIC_DEMO=True to enable model/tool calls.")